In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model
from keras.layers import concatenate, Dense
from tensorflow.keras import regularizers, Model, Input

from tensorflow.keras.callbacks import Callback, EarlyStopping

## Simulation 1b

In [2]:
X_id = pd.read_excel('Large_Scale_PONG/new_filtered/data/words_encodings.xlsx', index_col=0, sheet_name="identity").values.astype(np.float32)
X_pos = pd.read_excel('Large_Scale_PONG/new_filtered/data/words_encodings.xlsx', index_col=0, sheet_name="position").values.astype(np.float32)

Y = pd.read_excel('Large_Scale_PONG/new_filtered/data/words_encodings.xlsx', index_col=0,sheet_name="onehot").values[0:14525,0:350].astype(np.float32)

in_id   = Input(shape=(35,),  name='in_id')
tower_id  = Dense(700, activation='sigmoid')(in_id)


in_pos  = Input(shape=(100,), name='in_pos')
tower_pos = Dense(700, activation="sigmoid")(in_pos)



combined = concatenate([tower_id, tower_pos])
h = Dense(500, activation="sigmoid")(combined)
output = Dense(350, activation="sigmoid")(h)

model1 = Model(inputs=[in_id, in_pos], outputs=output)


model1.compile(optimizer="adam", loss="mse")
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10)

model1.fit([X_id, X_pos],Y,batch_size=25, epochs=700, callbacks=[callback])
# model1.save('simulation_1a_model.h5')

Epoch 1/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 22s 19ms/step - loss: 0.0235
Epoch 2/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - loss: 0.0147
Epoch 3/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - loss: 0.0139
Epoch 4/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - loss: 0.0133
Epoch 5/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - loss: 0.0122
Epoch 6/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - loss: 0.0113
Epoch 7/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - loss: 0.0104
Epoch 8/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - loss: 0.0099
Epoch 9/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - loss: 0.0093
Epoch 10/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - loss: 0.0089
Epoch 11/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - loss: 0.0084
Epoch 12/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - loss: 0.0080
Epoch 13/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0076
Epoch 14/700
577/577 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - loss: 0.0072
Epoch 15/700
577

# Test the model
We will find the correlation distance between the original word and its different modified forms.

In [3]:
from pathlib import Path

data_lst = [
    'Large_Scale_PONG/new_filtered/data/TL_words_encodings.xlsx',
    'Large_Scale_PONG/new_filtered/data/DL_words_encodings.xlsx'
]

df_sim1 = pd.DataFrame()
preds = {}

for doc in data_lst:
    key = Path(doc).stem.replace('_words_encodings', '')

    X_id = pd.read_excel(doc, sheet_name='identity', index_col=0).values.astype(np.float32)
    X_pos = pd.read_excel(doc, sheet_name='position', index_col=0).values.astype(np.float32)

    pred = model1.predict([X_id, X_pos])
    preds[key] = pred

    valz = []
    for i in range(len(Y)):
        r = np.corrcoef(pred[i], Y[i])[0, 1]
        valz.append(round(1 - r, 3))

    df_sim1[f'target-{key}'] = valz

df_sim1.to_excel('Large_Scale_PONG/new_filtered/results/PONG_simulation_1b_correlation_2.xlsx',index=False)

451/451 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step
451/451 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step
